In [8]:
import streamlit as st 
import requests
import json
from pprint import pprint
import pandas as pd 
import plotly.express as px

In [9]:
with open('../secrets.json') as f:
    secrets = json.load(f)

api_key = secrets['openaq-api-key']


In [10]:
def get_parameters(api_key):
    url = "https://api.openaq.org/v3/parameters"
    headers = { "X-API-Key": api_key}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data['results']
    else:
        return None
df_parameters = pd.DataFrame(get_parameters(api_key))
print("Parameter Definitions: ")
print(df_parameters)


Parameter Definitions: 
       id              name          units           displayName  \
0       1              pm10          µg/m³                  PM10   
1       2              pm25          µg/m³                 PM2.5   
2       3                o3          µg/m³               O₃ mass   
3       4                co          µg/m³               CO mass   
4       5               no2          µg/m³              NO₂ mass   
5       6               so2          µg/m³              SO₂ mass   
6       7               no2            ppm                   NO₂   
7       8                co            ppm                    CO   
8       9               so2            ppm                   SO₂   
9      10                o3            ppm                    O₃   
10     11                bc          µg/m³                    BC   
11     15               no2            ppb                   NO₂   
12     19               pm1          µg/m³                   PM1   
13     21               

In [11]:
from geopy.geocoders import Nominatim
import requests


def get_coordinates(address): # 1 request per sec is accepted
    geolocator = Nominatim(user_agent="air_quality_tracker")
    location = geolocator.geocode(address)
    if location:
        return location.latitude, location.longitude
    else:
        print("Address not found.")
        return None, None

def get_nearby_locations(lat,lon,api_key,radius,limit):    
    url = "https://api.openaq.org/v3/locations"
    headers = { "X-API-Key": api_key}
    params = {
        "coordinates": f"{lat},{lon}", 
        "radius": radius,                  
        "limit": limit
    }
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()
        return data['results']
    else:
        return None
    
    

In [12]:
def get_sensor_metadata(sensor_id, api_key):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}"
    headers = { "X-API-Key": api_key }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data['results']
    else:
        return {}

def get_sensors(location_id,api_key):
    url = f"https://api.openaq.org/v3/locations/{location_id}/sensors"
    headers = { "X-API-Key": api_key}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data['results']
    else:
        return None


def get_daily_measurements(sensor_id,api_key,date_from,date_to):
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"
    headers = {"X-API-Key": api_key}
    params = {
        "datetime_from": date_from , #format = YYYY-MM-DDTHH:MM:SSZ
        "datetime_to": date_to, #format = YYYY-MM-DDTHH:MM:SSZ
        "limit": 100,
    }
    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        data = response.json()
        return data['results']
    else:
        print(f"Request failed with status code {response.status_code}")


In [13]:
def get_latest_by_locationid(location_id, api_key):
    url = f"https://api.openaq.org/v3/locations/{location_id}/latest"
    headers = {"X-API-Key": api_key}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print('failed')
        return None



In [14]:
address = "Hyderabad, Telangana, India"
lat, lon = get_coordinates(address)

locations = get_nearby_locations(lat, lon, api_key, radius=10000, limit=1)
location_id = locations[0]['id']

df_parameters = pd.DataFrame(get_parameters(api_key))
sensors = get_sensors(location_id, api_key)
sensor_id_to_param = {s['id']: s['parameter'] for s in sensors}
latest = get_latest_by_locationid(location_id, api_key)
df_latest = pd.json_normalize(latest['results'])
df_latest['utc'] = df_latest['datetime.utc']
df_latest['local'] = df_latest['datetime.local']
df_latest['parameter_obj'] = df_latest['sensorsId'].map(sensor_id_to_param)

df_latest['parameter'] = df_latest['parameter_obj'].apply(
    lambda x: x['name'] if isinstance(x, dict) and 'name' in x else None
)


df_latest = df_latest[['parameter', 'value', 'utc', 'local', 'sensorsId']]
df_latest.dropna(subset=['parameter','value'], inplace=True)

print("Data to be plotted:")
print(df_latest[['parameter', 'value']])

if not df_latest.empty:
    fig = px.bar(df_latest, x='parameter', y='value', title='Latest Air Quality by Parameter')
    fig.show()
else:
    print("No valid data to plot.")

date_from = "2025-06-15"
date_to = "2025-06-22"
print("\n--- Daily Trends for Each Sensor ---")

for sensor in sensors:
    sensor_id = sensor['id']
    param_info = sensor['parameter']
    display_name = param_info['displayName'] if isinstance(param_info, dict) else param_info

    print(f"\nSensor ID: {sensor_id}, Parameter: {display_name}")

    results = get_daily_measurements(sensor_id, api_key, date_from, date_to)
    print(f"\nRaw results for Sensor ID {sensor_id}:")
    print(results)

    if results:
        df_daily = pd.DataFrame(results)
        if 'period' in df_daily.columns:
            df_daily['utc'] = pd.to_datetime(df_daily['period'].apply(
                lambda p: p['datetimeFrom']['utc'] if isinstance(p, dict) and 'datetimeFrom' in p else None
            ))
            df_daily['value'] = pd.to_numeric(df_daily['value'], errors='coerce')
            df_daily.dropna(subset=['utc', 'value'], inplace=True)

            if not df_daily.empty:
                fig = px.line(df_daily, x='utc', y='value', title=f"{display_name} Daily Trend", markers=True)
                fig.show()
            else:
                print("No valid data to plot.")
        else:
            print("No 'datetime' in results.")
    else:
        print("No daily data found.")


Data to be plotted:
           parameter   value
0                no2   18.70
1                 o3   14.10
2                 co  920.00
3               pm10  126.00
4               pm25   83.00
5                so2    0.00
6                so2    4.40
7                 o3   23.20
8                 no    3.60
9   relativehumidity   61.00
10                co    1.99
11               no2    9.30
12       temperature   26.50
13              pm25   56.00
14              pm10   96.00



--- Daily Trends for Each Sensor ---

Sensor ID: 712, Parameter: NO₂ mass

Raw results for Sensor ID 712:
[]
No daily data found.

Sensor ID: 711, Parameter: O₃ mass

Raw results for Sensor ID 711:
[]
No daily data found.

Sensor ID: 710, Parameter: CO mass

Raw results for Sensor ID 710:
[]
No daily data found.

Sensor ID: 715, Parameter: PM10

Raw results for Sensor ID 715:
[]
No daily data found.

Sensor ID: 714, Parameter: PM2.5

Raw results for Sensor ID 714:
[]
No daily data found.

Sensor ID: 713, Parameter: SO₂ mass

Raw results for Sensor ID 713:
[]
No daily data found.

Sensor ID: 12235585, Parameter: SO₂

Raw results for Sensor ID 12235585:
[{'value': 3.77, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 101, 'name': 'so2', 'units': 'ppb', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:0


Sensor ID: 12235581, Parameter: O₃ mass

Raw results for Sensor ID 12235581:
[{'value': 23.3, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 20.2, 'q02': 20.968, 'q25': 22.55, 'median': 23.25, 'q75': 24.05, 'q98': 26.022, 'max': 26.4, 'avg': 23.316666666666666, 'sd': 1.229437616934094}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 23.7, 'flagInfo': {'hasFlags'


Sensor ID: 12235579, Parameter: NO

Raw results for Sensor ID 12235579:
[{'value': 4.05, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 24, 'name': 'no', 'units': 'ppb', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 2.9, 'q02': 3.1, 'q25': 3.8, 'median': 4.1, 'q75': 4.375, 'q98': 5.044, 'max': 5.4, 'avg': 4.0500000000000025, 'sd': 0.4569685161136396}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 3.74, 'flagInfo': {'hasFlags': False}, 'para


Sensor ID: 12235584, Parameter: RH

Raw results for Sensor ID 12235584:
[{'value': 75.3, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 98, 'name': 'relativehumidity', 'units': '%', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 61.0, 'q02': 62.78, 'q25': 68.25, 'median': 79.0, 'q75': 82.0, 'q98': 83.0, 'max': 84.0, 'avg': 75.31111111111112, 'sd': 7.297289699183371}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 77.9, 'flagInfo': {'hasFlags'


Sensor ID: 12235578, Parameter: CO

Raw results for Sensor ID 12235578:
[{'value': 1.18, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 102, 'name': 'co', 'units': 'ppb', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 0.61, 'q02': 0.6911999999999999, 'q25': 0.96, 'median': 1.13, 'q75': 1.32, 'q98': 2.1415999999999995, 'max': 2.39, 'avg': 1.1763333333333332, 'sd': 0.32528449086914674}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 1.17, 'flag


Sensor ID: 12235580, Parameter: NO₂

Raw results for Sensor ID 12235580:
[{'value': 8.93, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 15, 'name': 'no2', 'units': 'ppb', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 7.0, 'q02': 7.1, 'q25': 7.6, 'median': 8.850000000000001, 'q75': 9.674999999999999, 'q98': 12.209999999999999, 'max': 12.6, 'avg': 8.932222222222219, 'sd': 1.4075101754099866}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 7.5


Sensor ID: 12235586, Parameter: Temperature (C)

Raw results for Sensor ID 12235586:
[{'value': 33.8, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 100, 'name': 'temperature', 'units': 'c', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 30.7, 'q02': 30.956, 'q25': 33.8, 'median': 34.1, 'q75': 34.4, 'q98': 34.8, 'max': 34.9, 'avg': 33.834444444444436, 'sd': 0.9622260136648167}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 31.2, 'flagInfo': 


Sensor ID: 12235583, Parameter: PM2.5

Raw results for Sensor ID 12235583:
[{'value': 7.26, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 1.0, 'q02': 1.0, 'q25': 4.0, 'median': 6.0, 'q75': 9.0, 'q98': 15.0, 'max': 15.0, 'avg': 7.2555555555555555, 'sd': 4.046252562587966}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 9.46, 'flagInfo': {'hasFlags': False}, 'p


Sensor ID: 12235582, Parameter: PM10

Raw results for Sensor ID 12235582:
[{'value': 57.3, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1 day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-06-14T18:30:00Z', 'local': '2025-06-15T00:00:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:30:00Z', 'local': '2025-06-16T00:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 51.0, 'q02': 51.0, 'q25': 54.0, 'median': 56.0, 'q75': 59.0, 'q98': 65.0, 'max': 65.0, 'avg': 57.25555555555555, 'sd': 4.046252562587966}, 'coverage': {'expectedCount': 96, 'expectedInterval': '24:00:00', 'observedCount': 90, 'observedInterval': '22:30:00', 'percentComplete': 94.0, 'percentCoverage': 94.0, 'datetimeFrom': {'utc': '2025-06-14T19:00:00Z', 'local': '2025-06-15T00:30:00+05:30'}, 'datetimeTo': {'utc': '2025-06-15T18:15:00Z', 'local': '2025-06-15T23:45:00+05:30'}}}, {'value': 59.5, 'flagInfo': {'hasFlags': False},